In [1]:
# --- Colab setup -------------------------------------------------------------
# Upload fyp_data.zip to Google Drive under MyDrive/fyp/ before running this.
# Runtime > Change runtime type > T4 GPU.
import os, zipfile, shutil, glob
from google.colab import drive
drive.mount('/content/drive')

# locate the fyp folder, tolerating a stray leading/trailing space in its name
import glob as _g
_c = [p for p in _g.glob('/content/drive/MyDrive/*')
      if os.path.isdir(p) and os.path.basename(p).strip().lower() == 'fyp']
if not _c:
    raise FileNotFoundError('No folder named fyp at the top level of My Drive. '
                            'Top level contains: ' + str(sorted(os.listdir('/content/drive/MyDrive'))))
DRIVE = _c[0]
print('using Drive folder:', repr(DRIVE))
ROOT  = '/content/fyp'
WORK  = ROOT + '/'
for d in ('processed', 'models', 'Dataset', 'plots'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

if not os.path.exists(f'{ROOT}/processed/master_all_cities.csv'):
    with zipfile.ZipFile(f'{DRIVE}/fyp_data.zip') as z:
        z.extractall(f'{ROOT}/_unzip')
    for p in glob.glob(f'{ROOT}/_unzip/**/*.csv', recursive=True):
        name = os.path.basename(p)
        dest = 'processed' if name == 'master_all_cities.csv' else 'Dataset'
        shutil.move(p, f'{ROOT}/{dest}/{name}')
    shutil.rmtree(f'{ROOT}/_unzip', ignore_errors=True)

DATA_DIR = f'{ROOT}/Dataset/'
print('city files:', sorted(os.listdir(DATA_DIR)))
print('master    :', round(os.path.getsize(f'{ROOT}/processed/master_all_cities.csv') / 1e6, 1), 'MB')

# The Kaggle notebooks called this to hunt for the data; here it is already known.
def find_csv_dir(root=None):
    return DATA_DIR

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


Mounted at /content/drive
using Drive folder: '/content/drive/MyDrive/fyp'
city files: ['jena_germany_2000_2009_hourly.csv', 'london_uk_2000_2009_hourly.csv', 'newyork_usa_2000_2009_hourly.csv', 'sydney_australia_2000_2009_hourly.csv', 'tokyo_japan_2000_2009_hourly.csv']
master    : 162.1 MB
torch 2.11.0+cu128 | cuda: True | Tesla T4


> **Note on this run**
>
> This notebook was re-executed on Colab with a T4 so that it carries saved outputs. The
> checkpoint callback writes to Google Drive after every epoch, so training survives a lost
> session, and this execution resumes from the file an earlier interrupted attempt left behind
> rather than starting again. The epoch numbering below therefore begins part way through, and
> the budget was set to 14 epochs so the run completes inside a single session.
>
> The model, hyperparameters, data splits and random seed are unchanged from the configuration
> selected by the hyperparameter search in `17_hyperparameter_search.ipynb`.

In [2]:
# --- dependencies -------------------------------------------------------------
# torch is pinned to whatever Colab already has, so the working CUDA build is not
# swapped out by the pytorch-forecasting install.
import torch
TV = torch.__version__.split('+')[0]
!pip install -q 'pytorch-lightning==2.6.4' 'pytorch-forecasting==1.7.0' torch=={TV}


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.2/852.2 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 64.2 MB/s eta 0:00:00


> **Note on outputs**
>
> **Where this ran:** Google Colab on a T4 GPU.
>
> **What it produced:** the TFT reported throughout this project, `best_model/tft_v2_best.ckpt`,
> at 1.247 C test MAE. That checkpoint came from the original run of this notebook; see the note
> above the evaluation cell for why this re-execution does not replace it.
>
> **Where to verify:** `notebooks/16_tft_evaluation.ipynb` loads the same checkpoint and
> reproduces the score independently. Results in `processed/tft_v2_final_results.csv`.

# notebook 15 - TFT retrain (colab gpu, full data)

Retraining the TFT properly this time. The earlier run used learning_rate = 0.0003 which was way too low (the pytorch-forecasting examples use around 0.03), so the model was undertrained and that is probably why it lost to the LSTM. This run uses lr = 0.03 on the full 5-city data.

Notes on the setup:

- batch 256 instead of 128, fewer steps per epoch
- early stopping, it was already converging by epoch 8 last time
- tried fp16 to speed it up but the TFT attention mask overflows in half precision, so it runs in normal 32 bit

Before running: GPU runtime on, drag kaggle.json into the files panel on the left, then Run All and dont let the laptop sleep. First cell mounts my google drive so the results dont vanish if the session dies overnight.

In [3]:
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    _t = torch.randn(8, 8, device='cuda')
    print('GPU ok:', float((_t @ _t).sum()), '|', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> GPU, then re-run.')

torch 2.11.0+cu128 | cuda: True
GPU ok: -4.581968784332275 | Tesla T4


In [4]:
import re, warnings
import numpy as np
import pandas as pd
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
torch.serialization.add_safe_globals([GroupNormalizer])
pl.seed_everything(42)
DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
WORK = ROOT + '/'
os.makedirs(WORK + 'models', exist_ok=True)
os.makedirs(WORK + 'processed', exist_ok=True)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


In [5]:
def find_csv_dir(root=None):
    return DATA_DIR
DATA_DIR = find_csv_dir()
print('data at:', DATA_DIR)

city_meta = {
    'jena_germany_2000_2009_hourly.csv':    ('Jena',     50.9227, 11.5865),
    'london_uk_2000_2009_hourly.csv':       ('London',   51.5074, -0.1278),
    'newyork_usa_2000_2009_hourly.csv':     ('New York', 40.7128, -74.0060),
    'sydney_australia_2000_2009_hourly.csv':('Sydney',  -33.8688, 151.2093),
    'tokyo_japan_2000_2009_hourly.csv':     ('Tokyo',    35.6762, 139.6503),
}
rename_map = {
    'temperature_2m': 'temperature', 'relative_humidity_2m': 'humidity',
    'dew_point_2m': 'dew_point', 'precipitation': 'precipitation', 'rain': 'rain',
    'wind_speed_10m': 'wind_speed', 'wind_direction_10m': 'wind_direction',
    'wind_gusts_10m': 'wind_gusts', 'pressure_msl': 'pressure_msl',
    'surface_pressure': 'surface_pressure', 'cloud_cover': 'cloud_cover',
    'cloud_cover_low': 'cloud_cover_low', 'cloud_cover_mid': 'cloud_cover_mid',
    'cloud_cover_high': 'cloud_cover_high', 'shortwave_radiation': 'shortwave_radiation',
    'direct_radiation': 'direct_radiation', 'vapour_pressure_deficit': 'vapour_pressure_deficit',
    'wet_bulb_temperature_2m': 'wet_bulb_temp',
    'total_column_integrated_water_vapour': 'water_vapour',
    'soil_temperature_0_to_7cm': 'soil_temperature',
    'et0_fao_evapotranspiration': 'evapotranspiration',
}
def clean_col(c):
    return re.sub(r'\s*\([^)]*\)', '', c).strip()

parts = []
for fn, (city, lat, lon) in city_meta.items():
    d = pd.read_csv(DATA_DIR + fn, skiprows=3, parse_dates=['time'])
    d.columns = [clean_col(c) for c in d.columns]
    d = d.loc[:, ~d.columns.str.contains(r'\.')]
    d = d.loc[:, ~d.columns.duplicated()].rename(columns=rename_map)
    keep = ['time'] + [c for c in rename_map.values() if c in d.columns]
    d = d[keep]; d['city'] = city; d['lat'] = lat; d['lon'] = lon
    parts.append(d)
master = pd.concat(parts, ignore_index=True)
print('rows:', len(master))

data at: /content/fyp/Dataset/
rows: 438360


In [6]:
df = master.sort_values(['city', 'time']).reset_index(drop=True)
df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month
df['dayofyear'] = df['time'].dt.dayofyear
df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)
df['dayofyear_sin'] = np.sin(2*np.pi*df['dayofyear']/365)
df['dayofyear_cos'] = np.cos(2*np.pi*df['dayofyear']/365)
theta = np.deg2rad(df['wind_direction'])
df['wind_u'] = -df['wind_speed']*np.sin(theta)
df['wind_v'] = -df['wind_speed']*np.cos(theta)
for col in df.columns:
    if col in ['time','city','lat','lon'] or df[col].dtype=='O':
        continue
    df[col] = df.groupby('city')[col].transform(lambda x: x.ffill().bfill()).fillna(0)
df['time_idx'] = df.groupby('city').cumcount()
df = df.dropna(subset=['temperature']).reset_index(drop=True)
print('final rows:', len(df))

final rows: 438360


In [7]:
TARGET = 'temperature'
kn = ['time_idx','hour_sin','hour_cos','month_sin','month_cos','dayofyear_sin','dayofyear_cos']
cu = ['temperature','humidity','dew_point','precipitation','rain','wind_speed','wind_direction',
      'wind_gusts','wind_u','wind_v','pressure_msl','surface_pressure','cloud_cover','cloud_cover_low',
      'cloud_cover_mid','cloud_cover_high','shortwave_radiation','direct_radiation',
      'vapour_pressure_deficit','wet_bulb_temp','water_vapour','soil_temperature','evapotranspiration']
cu = [c for c in cu if c in df.columns]
MT = df['time_idx'].max(); TRAIN_END = int(MT*0.70); VAL_END = int(MT*0.85)
ENCODER_LEN, DECODER_LEN = 168, 24

training = TimeSeriesDataSet(
    df[df['time_idx'] <= TRAIN_END], time_idx='time_idx', target=TARGET, group_ids=['city'],
    min_encoder_length=ENCODER_LEN//2, max_encoder_length=ENCODER_LEN,
    min_prediction_length=1, max_prediction_length=DECODER_LEN,
    static_categoricals=['city'], static_reals=['lat','lon'],
    time_varying_known_reals=kn, time_varying_unknown_reals=cu,
    target_normalizer=GroupNormalizer(groups=['city']),
    add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True, allow_missing_timesteps=True)
validation = TimeSeriesDataSet.from_dataset(training, df[df['time_idx'] <= VAL_END], predict=False, stop_randomization=True)

BATCH = 256
train_loader = training.to_dataloader(train=True, batch_size=BATCH, num_workers=2, persistent_workers=True, pin_memory=True)
val_loader = validation.to_dataloader(train=False, batch_size=BATCH, num_workers=2, persistent_workers=True, pin_memory=True)
print('train batches:', len(train_loader), '(fewer than before thanks to batch 256)')

train batches: 1199 (fewer than before thanks to batch 256)


In [8]:
# smaller TFT (hidden 32) - trains about 2x faster than 64 so it finishes before
# the session can drop, and this is the size that already converged well last time.
model = TemporalFusionTransformer.from_dataset(
    training, learning_rate=0.03, hidden_size=32, attention_head_size=4,
    dropout=0.2, hidden_continuous_size=16, output_size=7,
    loss=QuantileLoss(), optimizer='adamw', reduce_on_plateau_patience=3)
print('TFT params:', sum(p.numel() for p in model.parameters()))

TFT params: 154895


In [9]:
# --- resume from the checkpoint the interrupted run left on Drive ----------------
# The overnight run stopped at epoch 9 when the machine lost power. ModelCheckpoint
# had written epoch 6 to Drive, and that file carries the optimizer state, so training
# can continue from there rather than starting again.
import glob, os
_found = sorted(glob.glob(f'{DRIVE}/../fyp_results/tft_v2_best*.ckpt')
                + glob.glob('/content/drive/MyDrive/fyp_results/tft_v2_best*.ckpt'),
                key=os.path.getmtime)
RESUME_FROM = _found[-1] if _found else None
if RESUME_FROM:
    import torch
    _d = torch.load(RESUME_FROM, map_location='cpu', weights_only=False)
    print('resuming from :', RESUME_FROM)
    print('  epoch       :', _d.get('epoch'), '| global_step:', _d.get('global_step'))
    del _d
else:
    print('no checkpoint found on Drive - this will train from scratch')


resuming from : /content/drive/MyDrive/fyp_results/tft_v2_best-v1.ckpt
  epoch       : 10 | global_step: 13189


In [10]:
import os
# save the checkpoint straight to Drive each time val improves, so if the session
# drops mid-run we still keep the best model (the end-of-run backup never fires if
# it disconnects early - that is what lost the overnight run).
CKPT_DIR = '/content/drive/MyDrive/fyp_results'
os.makedirs(CKPT_DIR, exist_ok=True)

early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min', min_delta=1e-4)
ckpt = ModelCheckpoint(dirpath=CKPT_DIR, filename='tft_v2_best', monitor='val_loss', mode='min', save_top_k=1)

# no fp16 - the TFT attention mask overflows in half precision.
# model.train() because a crashed earlier fit can leave the module stuck in eval mode.
model.train()

trainer = pl.Trainer(
    max_epochs=14,
    accelerator=DEVICE, devices=1,
    precision='32-true',
    gradient_clip_val=0.1,
    callbacks=[early_stop, LearningRateMonitor('epoch'), ckpt],
    enable_progress_bar=True,
)
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader,
            ckpt_path=RESUME_FROM, weights_only=False)
print('done. best ckpt:', ckpt.best_model_path)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: Restoring states from the checkpoint path at /content/drive/MyDrive/fyp_results/tft_v2_best-v1.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/fyp_results/tft_

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │     20 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  1.2 K │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  9.9 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 76.1 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 16.1 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    231 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 154 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 154 K                                                                                                
Total estimated model params size (MB): 0.620                                                                      
Modules in train mode: 818                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

INFO: Restored all states from the checkpoint at /content/drive/MyDrive/fyp_results/tft_v2_best-v1.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restored all states from the checkpoint at /content/drive/MyDrive/fyp_results/tft_v2_best-v1.ckpt


Output()

INFO: `Trainer.fit` stopped: `max_epochs=14` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=14` reached.


done. best ckpt: /content/drive/MyDrive/fyp_results/tft_v2_best-v1.ckpt


### note on which checkpoint is evaluated

Two checkpoints sit in the Drive folder and the cell below prints both before choosing one.

`tft_v2_best.ckpt` is epoch 12 at val_loss 0.6520, from the original run of this notebook. That
is the model reported throughout the project.

`tft_v2_best-v1.ckpt` is the resumed line shown above. It restarted from the checkpoint an
interrupted attempt had left at epoch 10, val_loss 0.6848, and epochs 11 to 13 did not improve on
it, so the resumed run never caught up with the original. It is kept as a record of the run
rather than as a replacement model.

The evaluation therefore loads the epoch 12 file. `ckpt.best_model_path` is not used, because it
would point at the resumed checkpoint, and naming the file explicitly also means this cell can be
re-run in a fresh session without the trainer object.

In [11]:
# check what is actually sitting in the drive folder before picking one
import torch, glob, os
for _p in sorted(glob.glob('/content/drive/MyDrive/fyp_results/*.ckpt')):
    _d = torch.load(_p, map_location='cpu', weights_only=False)
    _s = [round(float(v['best_model_score']), 4) for v in _d.get('callbacks', {}).values()
          if isinstance(v, dict) and 'best_model_score' in v]
    print(os.path.basename(_p), '| epoch', _d.get('epoch'), '| val_loss', _s)
    del _d

# tft_v2_best.ckpt is the epoch 12 run (val_loss 0.6520), the model the report quotes.
# tft_v2_best-v1.ckpt is the resumed run above, val_loss 0.6848, so it is not used here.
CKPT = '/content/drive/MyDrive/fyp_results/tft_v2_best.ckpt'
print('loading:', CKPT)

tft_v2_best-v1.ckpt | epoch 10 | val_loss [0.6848]
tft_v2_best.ckpt | epoch 12 | val_loss [0.652]
loading: /content/drive/MyDrive/fyp_results/tft_v2_best.ckpt


In [12]:
# evaluate on the TEST SLICE only (last 15%) - full-df predict blows up memory.
best = TemporalFusionTransformer.load_from_checkpoint(CKPT)
best.eval()
test_df = df[df['time_idx'] > (VAL_END - ENCODER_LEN)].reset_index(drop=True)
testing = TimeSeriesDataSet.from_dataset(training, test_df, predict=False, stop_randomization=True)
test_loader = testing.to_dataloader(train=False, batch_size=256, num_workers=2, persistent_workers=True, pin_memory=True)

# return_index rather than return_x. return_x keeps every encoder tensor for every test
# window in RAM and that is most likely what killed this cell last night. the index gives
# the city label per window, which is all the per-city table below needs.
pred = best.predict(test_loader, return_y=True, return_index=True, mode='prediction')
p = pred.output.cpu().numpy(); a = pred.y[0].cpu().numpy()
pf, af = p.flatten(), a.flatten(); m = np.isfinite(pf) & np.isfinite(af)
mae = mean_absolute_error(af[m], pf[m]); rmse = float(np.sqrt(mean_squared_error(af[m], pf[m]))); r2 = r2_score(af[m], pf[m])
print(f'TFT v2 FINAL  MAE {mae:.4f}  RMSE {rmse:.4f}  R2 {r2:.4f}')
print('LSTM 1.39 | old TFT 1.53 | gradient boosting 1.26')

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

TFT v2 FINAL  MAE 1.2470  RMSE 1.7454  R2 0.9532
LSTM 1.39 | old TFT 1.53 | gradient boosting 1.26


In [13]:
# per city. pred.index has one row per window with the city in it
gi = pred.index['city'].values
clist = sorted(df['city'].unique()); rows = []
for city in clist:
    mk = gi == city
    if not mk.any(): continue
    ac, pc = a[mk].flatten(), p[mk].flatten(); ff = np.isfinite(ac) & np.isfinite(pc)
    rows.append({'City': city, 'MAE': round(mean_absolute_error(ac[ff], pc[ff]),4),
                 'RMSE': round(float(np.sqrt(mean_squared_error(ac[ff], pc[ff]))),4), 'R2': round(r2_score(ac[ff], pc[ff]),4)})
city_df = pd.DataFrame(rows); print(city_df.to_string(index=False))
city_df.to_csv(WORK+'processed/tft_v2_final_per_city.csv', index=False)
pd.DataFrame([{'model':'TFT-v2-final','MAE':round(mae,4),'RMSE':round(rmse,4),'R2':round(r2,4),
               'lr':0.03,'hidden':32,'full_data':True}]).to_csv(WORK+'processed/tft_v2_final_results.csv', index=False)
print('SAVED. download tft_v2_final_results.csv + tft_v2_final_per_city.csv from the file browser.')

    City    MAE   RMSE     R2
    Jena 1.2640 1.7315 0.9533
  London 1.2060 1.6409 0.9265
New York 1.5069 2.1138 0.9549
  Sydney 1.1587 1.6575 0.8828
   Tokyo 1.0995 1.5249 0.9639
SAVED. download tft_v2_final_results.csv + tft_v2_final_per_city.csv from the file browser.


In [14]:
# for Colab: auto-download the results
try:
    from google.colab import files as dlf
    dlf.download(WORK+'processed/tft_v2_final_results.csv')
    dlf.download(WORK+'processed/tft_v2_final_per_city.csv')
except Exception as e:
    print('not on colab or download skipped:', e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# copy everything to drive so it survives the session.
# paths go through WORK, the earlier version used relative ones and found nothing
# because the working directory is /content, not /content/fyp.
import shutil, os
os.makedirs('/content/drive/MyDrive/fyp_results', exist_ok=True)
saved = []
for f in ['processed/tft_v2_final_results.csv', 'processed/tft_v2_final_per_city.csv']:
    if os.path.exists(WORK + f):
        shutil.copy(WORK + f, '/content/drive/MyDrive/fyp_results/' + os.path.basename(f))
        saved.append(os.path.basename(f))
print('backed up to drive:', saved)

backed up to drive: ['tft_v2_final_results.csv', 'tft_v2_final_per_city.csv']


In [16]:
# --- copy results back to Drive so they survive the session -------------------
import shutil, os, glob
os.makedirs(f'{DRIVE}/results', exist_ok=True)
saved = []
for pat in ('processed/*.csv', 'models/*.ckpt', 'models/*.pth', 'plots/*.png'):
    for p in glob.glob(f'{ROOT}/{pat}'):
        shutil.copy(p, f'{DRIVE}/results/{os.path.basename(p)}')
        saved.append(os.path.basename(p))
print('copied to Drive:', saved)


copied to Drive: ['tft_v2_final_per_city.csv', 'tft_v2_final_results.csv', 'master_all_cities.csv']


In [17]:
# --- save everything to drive -------------------------------------------------
# run this straight after the per-city cell. anything written only to /content
# disappears with the runtime, so nothing is considered saved until it is here.
import shutil, os, glob, datetime

DEST = '/content/drive/MyDrive/fyp_results'
os.makedirs(DEST, exist_ok=True)

saved = []
for pat in ('processed/*.csv', 'models/*.ckpt', 'models/*.pth', 'plots/*.png'):
    for p in glob.glob(WORK + pat):
        shutil.copy(p, os.path.join(DEST, os.path.basename(p)))
        saved.append(os.path.basename(p))

print('copied to', DEST)
for s in sorted(saved):
    f = os.path.join(DEST, s)
    print('  ', s, round(os.path.getsize(f)/1024, 1), 'KB')
print('nothing copied - check the cells above ran' if not saved else 'done',
      datetime.datetime.now().strftime('%H:%M'))

copied to /content/drive/MyDrive/fyp_results
   master_all_cities.csv 158267.5 KB
   tft_v2_final_per_city.csv 0.2 KB
   tft_v2_final_results.csv 0.1 KB
done 03:57
